In [8]:
"""
NSE Holiday Calendar Generator

Fetches trading holidays from the official NSE India API (Cash Market segment)
and generates a clean CSV calendar that includes:
- All weekends (Saturday & Sunday)
- All NSE trading holidays

The output CSV contains only non-trading days with columns:
    Date, Day, Type, Holiday

Output is saved to: ../data/input/nse_holidays.csv
"""

import requests
import pandas as pd
import time


# =============================================================================
# Configuration
# =============================================================================

# NSE API endpoint for trading holidays
NSE_HOLIDAY_URL = "https://www.nseindia.com/api/holiday-master?type=trading"

# Headers required to avoid NSE blocking the request
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://www.nseindia.com/",
}

# Output path for the generated calendar
OUTPUT_CSV_PATH = "../data/input/nse_holidays.csv"


def fetch_nse_holidays() -> dict:
    """Fetch raw holiday data from the NSE India API.

    This function establishes a session, retrieves necessary cookies from the
    homepage, then calls the holiday master API.

    Returns:
        dict: JSON response containing holiday lists for different market segments
              (CM, FO, etc.).

    Raises:
        Exception: If the HTTP request does not return status 200.
    """
    session = requests.Session()
    session.headers.update(HEADERS)

    # Step 1: Visit homepage to obtain session cookies
    session.get("https://www.nseindia.com")
    time.sleep(1)  # Small delay to mimic browser behaviour

    # Step 2: Fetch holiday data
    response = session.get(NSE_HOLIDAY_URL)

    if response.status_code != 200:
        print(f"Status: {response.status_code}")
        print(response.text[:200])
        raise Exception("❌ NSE blocked request or API error")

    return response.json()


def create_holiday_calendar() -> None:
    """Generate and save the complete NSE holiday calendar for the year.

    1. Fetches holiday data via the NSE API.
    2. Extracts Cash Market (CM) holidays.
    3. Builds a full-year DataFrame.
    4. Merges weekends and holidays.
    5. Filters to non-trading days only.
    6. Saves the result as CSV.

    The year is determined dynamically from the first holiday date returned by
    the API (NSE typically returns data for a single calendar year).
    """
    # Fetch raw data from NSE
    data = fetch_nse_holidays()

    # Extract Cash Market (CM) holidays
    holidays_df = pd.DataFrame(data["CM"])
    holidays_df = holidays_df[["tradingDate", "description"]]
    holidays_df.columns = ["Date", "Holiday"]

    # Convert string dates to datetime
    holidays_df["Date"] = pd.to_datetime(
        holidays_df["Date"], format="%d-%b-%Y"
    )

    # Determine the year from the fetched data
    year = holidays_df["Date"].dt.year.iloc[0]

    # Create a complete calendar for the entire year
    full_calendar = pd.DataFrame({
        "Date": pd.date_range(
            start=f"{year}-01-01",
            end=f"{year}-12-31",
        )
    })

    # Add day name and weekend flag
    full_calendar["Day"] = full_calendar["Date"].dt.day_name()
    full_calendar["Is_Weekend"] = full_calendar["Day"].isin(
        ["Saturday", "Sunday"]
    )

    # Merge NSE holidays into the calendar
    calendar = full_calendar.merge(holidays_df, on="Date", how="left")

    # Keep only weekends and official holidays
    calendar = calendar[
        calendar["Is_Weekend"] | calendar["Holiday"].notna()
    ].copy()

    # Create a clean "Type" column
    calendar["Type"] = calendar.apply(
        lambda row: "Weekend" if row["Is_Weekend"] else "Holiday",
        axis=1,
    )

    # Final column selection and sorting
    calendar = calendar[["Date", "Day", "Type", "Holiday"]]
    calendar = calendar.sort_values("Date").reset_index(drop=True)

    # Save to CSV
    calendar.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"✅ NSE holiday calendar successfully saved to: {OUTPUT_CSV_PATH}")


if __name__ == "__main__":
    create_holiday_calendar()

✅ NSE holiday calendar successfully saved to: ../data/input/nse_holidays.csv
